In [1]:
import logging

from nichejepa.datasets.cell_neighborhood_dataset import make_cell_neighborhood_dataset
from nichejepa.masks.multigene import MaskCollator
from nichejepa.utils.config_utils import create_params_from_YAML_wandb_config, prepare_dataset

In [2]:
world_size = 1
rank = 0

In [3]:
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)
params = create_params_from_YAML_wandb_config('../../../nichejepa/configs/test.yaml',
                                              logger)
params["data"]["batch_size"] = 2

INFO:root:Loaded parameters from YAML file.


In [4]:
params

{'data': {'batch_size': 2,
  'data_path': '/lustre/scratch126/cellgen/team361/mv10/nichejepa-reproducibility/nichejepa-reproducibility/datasets/st_data/gold/cell_neighborhood_tokenizer/merfish_mouse_brain_subset/merfish_mouse_brain_subset_read_depth_shifted_log_2170.dataset/',
  'data_set_name': 'merfish',
  'num_workers': 1,
  'pin_mem': False,
  'seq_len': 2000,
  'vocab_size': 1087,
  'incl_cell_seq': True,
  'incl_neighborhood_seq': True,
  'seq_len_cell': 1000,
  'seq_len_neighborhood': 1000,
  'has_cls': True,
  'sample_size': 100000,
  'sample_subset': True,
  'random_state': 0,
  'stratify': False,
  'split': 0.2,
  'specific_cell_types': []},
 'emb': {'weighted_average': False,
  'average': True,
  'cls': True,
  'select_topk': False,
  'retrieve_gene': False,
  'retrieve_niche': False,
  'retrieve_cell': True,
  'gene_id': 332,
  'retrieve_position_emb': False,
  'retrieve_emb_from_layer': 2},
 'logging': {'folder': 'logs/merfish_200', 'write_tag': 'jepa'},
 'mask': {'context

In [5]:
train_dataset, _ = prepare_dataset(params)
train_dataset

Loading dataset from disk:   0%|          | 0/52 [00:00<?, ?it/s]

Dataset({
    features: ['gene_tokens_cell', 'gene_tokens_neighborhood', 'cell_types', 'niche_types', 'donor_id', 'sample_id', 'batch_id', 'input_ids'],
    num_rows: 80000
})

In [6]:
mask_collator = MaskCollator(
    n_targets=params["mask"]["n_targets"],
    n_contexts=params["mask"]["n_contexts"],
    target_mask_size=params["mask"]["target_mask_size"],
    context_mask_size=params["mask"]["context_mask_size"],
    seq_len_cell=params["data"]["seq_len_cell"],
    seq_len_neighborhood=params["data"]["seq_len_neighborhood"],
    has_cls=params["emb"]["cls"])

In [7]:
_, train_loader, train__sampler = make_cell_neighborhood_dataset(
            batch_size=params["data"]["batch_size"],
            data=train_dataset,
            vocab_size=params["data"]["vocab_size"],
            seq_len=params["data"]["seq_len"],
            collator=mask_collator,
            pin_mem=params["data"]["pin_mem"],
            num_workers=params["data"]["num_workers"],
            world_size=world_size,
            rank=rank,
            drop_last=False,
            incl_cell_seq=params["data"]["incl_cell_seq"],
            incl_neighborhood_seq=params["data"]["incl_neighborhood_seq"],
            seq_len_cell=params["data"]["seq_len_cell"],
            seq_len_neighborhood=params["data"]["seq_len_neighborhood"],
            has_cls=params["emb"]["cls"])

INFO:root:Data loader created.


In [8]:
for itr, (udata, masks_enc, masks_pred) in enumerate(train_loader):
    print(udata)
    print(masks_enc)
    print(masks_pred)
    break

[tensor([[1087,  635,  341,  ...,    0,    0,    0],
        [1087,  236,  830,  ...,    0,    0,    0]]), tensor([[1, 1, 1,  ..., 2, 2, 2],
        [1, 1, 1,  ..., 2, 2, 2]], dtype=torch.int32), ('white matter', 'brain'), ('oligodendrocyte', 'vascular leptomeningeal cell')]
[tensor([[   0, 1559, 1560],
        [   0, 1725, 1726]])]
[tensor([[   0, 1036, 1037, 1038, 1039],
        [   0, 1099, 1100, 1101, 1102]]), tensor([[   0, 1055, 1056, 1057, 1058],
        [   0,   40,   41,   42,   43]])]
